# Capstone: one model through the whole lifecycle

This is the **capstone**. Every notebook before it taught one feature in isolation, each (in the official docs) on a *different* toy dataset. The gap that leaves — and the thing this notebook closes — is the **narrative thread**: there is no upstream tutorial that follows a *single* model from raw data all the way to a live endpoint, making explicit the **decisions** that connect the stages.

So this notebook does not re-teach any single API. It **reuses** them and spends its words on the joins between steps:

> feature-engineer → **tune** a few candidates → **evaluate** them → **gate** the winner → **register & promote** it → **load** it → **serve** it.

Each step links back to the notebook that taught it. If a step feels thin here, that's deliberate — the depth lives in `c_`/`f_`/`g_`/`h_`/`i_`/`j_`; the capstone's job is to show them working *as one pipeline* on **one** dataset (California housing + tree models, for continuity with `d_`–`j_`).

**Prerequisites:** the tracking server on `127.0.0.1:5001` (repo root, as in `f_`–`j_`), and — for the serving step — a free port `5002`. No new dependencies beyond what `i_`/`j_` already added.

## Setup

One experiment for the whole run, and a capstone-specific registered-model name (`ca_housing_capstone`) so we don't collide with `g_`/`h_`'s `ca_housing_price`.

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

import mlflow
from mlflow import MlflowClient
from mlflow.exceptions import RestException
from mlflow.models import MetricThreshold, infer_signature

TRACKING_URI = "http://127.0.0.1:5001"
SERVE_URL = "http://127.0.0.1:5002"
MODEL_NAME = "ca_housing_capstone"
TARGET = "MedHouseVal"

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

EXPERIMENT = "Capstone: End-to-End"
try:
    mlflow.create_experiment(EXPERIMENT)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(EXPERIMENT)
print("tracking:", TRACKING_URI, "| experiment:", EXPERIMENT)

tracking: http://127.0.0.1:5001 | experiment: Capstone: End-to-End


## Step 1 — Feature-engineer, and log the data lineage  ·  *reuses `i_`*

We derive a few features, then record **both** the raw source and the engineered feature set on the run with `mlflow.log_input` (the pattern from `i_dataset_logging`). The decision this encodes: *the model trains on engineered features, so the run must say which features, derived from what.* We write both frames to disk so their `source` is genuinely re-loadable.

In [2]:
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["rooms_per_person"] = out["AveRooms"] / out["AveOccup"]
    out["bedrooms_per_room"] = out["AveBedrms"] / out["AveRooms"]
    out["log_median_income"] = np.log1p(out["MedInc"])
    return out


DEMO_DIR = Path("_capstone_demo")
DEMO_DIR.mkdir(exist_ok=True)
RAW_CSV, ENG_CSV = DEMO_DIR / "raw.csv", DEMO_DIR / "engineered_v1.csv"

raw_df = fetch_california_housing(as_frame=True).frame
eng_df = engineer(raw_df)
raw_df.to_csv(RAW_CSV, index=False)
eng_df.to_csv(ENG_CSV, index=False)

raw_ds = mlflow.data.from_pandas(raw_df, source=str(RAW_CSV), name="ca-housing-raw", targets=TARGET)
eng_ds = mlflow.data.from_pandas(eng_df, source=str(ENG_CSV), name="ca-housing-engineered-v1", targets=TARGET)

feature_cols = [c for c in eng_df.columns if c != TARGET]
X = eng_df[feature_cols]
y = eng_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
eval_test = X_test.copy()
eval_test[TARGET] = y_test  # evaluate() wants features + target in one frame

print(f"{len(feature_cols)} engineered features:", feature_cols)
print("train/test:", X_train.shape, X_test.shape)

11 engineered features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'rooms_per_person', 'bedrooms_per_room', 'log_median_income']
train/test: (15480, 11) (5160, 11)


/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


## Step 2 — Tune a few candidates as child runs  ·  *reuses `c_` + `j_`*

A parent run with one child per candidate (the parent/child shape from `c_`). We also flip on **`log_system_metrics=True`** for the parent — the `j_` integration — so the tuning step carries a resource story. *(California housing is small, so the system charts here are modest; `j_` is where they really move. The point is that turning it on is one keyword.)*

Each child logs its model **and** the dataset lineage from Step 1, so every candidate is independently registrable and auditable.

In [3]:
mlflow.set_system_metrics_sampling_interval(1)

candidates = [
    {"name": "rf_shallow", "params": {"n_estimators": 100, "max_depth": 6}},
    {"name": "rf_mid",     "params": {"n_estimators": 200, "max_depth": 12}},
    {"name": "rf_deep",    "params": {"n_estimators": 300, "max_depth": 20}},
]

results = []
with mlflow.start_run(run_name="tuning", log_system_metrics=True) as parent:
    parent_id = parent.info.run_id
    for cand in candidates:
        with mlflow.start_run(run_name=cand["name"], nested=True) as child:
            mlflow.log_input(raw_ds, context="raw_source")
            mlflow.log_input(eng_ds, context="training", tags={"feature_set": "v1"})
            mlflow.log_params(cand["params"])

            model = RandomForestRegressor(random_state=0, n_jobs=-1, **cand["params"])
            model.fit(X_train, y_train)
            signature = infer_signature(X_train, model.predict(X_train))
            info = mlflow.sklearn.log_model(
                model, name="model", signature=signature, input_example=X_train.head()
            )
            results.append({"name": cand["name"], "run_id": child.info.run_id,
                            "model_uri": info.model_uri})
            print(f"  logged {cand['name']:11} -> {info.model_uri}")

print("parent run:", parent_id)

2026/06/02 14:58:21 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


2026/06/02 14:58:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/02 14:58:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  logged rf_shallow  -> models:/m-835921281b22416b9144d11b56d3bb66
🏃 View run rf_shallow at: http://127.0.0.1:5001/#/experiments/8/runs/998c5f523d614532ad42f047d76377e7
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8


2026/06/02 14:58:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/06/02 14:58:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


  logged rf_deep     -> models:/m-fff98a9d4dd24d4d93af486ad5d1ccad
🏃 View run rf_deep at: http://127.0.0.1:5001/#/experiments/8/runs/a40030671e044beaabae85dc52dc2c6f
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8
🏃 View run tuning at: http://127.0.0.1:5001/#/experiments/8/runs/6a893d198e474e908730897bc8e1df35
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8


2026/06/02 14:58:35 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


2026/06/02 14:58:35 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


parent run: 6a893d198e474e908730897bc8e1df35


## Step 3 — Evaluate every candidate on held-out data  ·  *reuses `f_`*

`mlflow.models.evaluate(...)` loads each logged model back and scores it against the *same* held-out frame. Evaluating from the **logged artifact** (not the in-memory object) is the point: it's the artifact we'll ship. We collect RMSE/R² so the next step can pick on evidence.

In [4]:
for r in results:
    with mlflow.start_run(run_name=f"eval_{r['name']}"):
        res = mlflow.models.evaluate(
            model=r["model_uri"], data=eval_test, targets=TARGET,
            model_type="regressor", evaluators=["default"],
        )
    r["rmse"] = res.metrics["root_mean_squared_error"]
    r["r2"] = res.metrics["r2_score"]

leaderboard = sorted(results, key=lambda r: r["rmse"])
print(f"{'candidate':12} {'RMSE':>8} {'R2':>8}")
for r in leaderboard:
    print(f"{r['name']:12} {r['rmse']:8.4f} {r['r2']:8.4f}")

winner, runner_up = leaderboard[0], leaderboard[1]
print(f"\nwinner: {winner['name']}  |  runner-up: {runner_up['name']}")

2026/06/02 14:58:36 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-835921281b22416b9144d11b56d3bb66


2026/06/02 14:58:36 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


2026/06/02 14:58:36 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


2026/06/02 14:58:36 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


🏃 View run eval_rf_shallow at: http://127.0.0.1:5001/#/experiments/8/runs/9ffe664f28b44bd8862b2d265c5b40a8
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8


...

2026/06/02 14:58:38 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-fff98a9d4dd24d4d93af486ad5d1ccad


2026/06/02 14:58:38 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


2026/06/02 14:58:38 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


2026/06/02 14:58:38 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


🏃 View run eval_rf_deep at: http://127.0.0.1:5001/#/experiments/8/runs/43802934474d4b1697af45fd35629821
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8
candidate        RMSE       R2
rf_deep        0.5199   0.7955
rf_mid         0.5369   0.7819
rf_shallow     0.6550   0.6754

winner: rf_deep  |  runner-up: rf_mid


## Step 4 — Gate the winner against a baseline  ·  *reuses `f_`*

"Best of three" isn't good enough to ship — it could still be worse than a trivial predictor. So we evaluate a `DummyRegressor` (predict-the-mean) baseline and require the winner to **beat it by a real margin** with `mlflow.validate_evaluation_results(...)`. This is the decision the official docs leave implicit: *a candidate earns promotion only by clearing a bar, not by winning a beauty contest.*

In [5]:
# Baseline: predict the training mean.
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
with mlflow.start_run(run_name="baseline_eval") as brun:
    bsig = infer_signature(X_train, baseline.predict(X_train))
    binfo = mlflow.sklearn.log_model(baseline, name="model", signature=bsig,
                                     input_example=X_train.head())
    baseline_result = mlflow.models.evaluate(
        model=binfo.model_uri, data=eval_test, targets=TARGET,
        model_type="regressor", evaluators=["default"],
    )

with mlflow.start_run(run_name="winner_eval") as wrun:
    winner_result = mlflow.models.evaluate(
        model=winner["model_uri"], data=eval_test, targets=TARGET,
        model_type="regressor", evaluators=["default"],
    )

print(f"baseline RMSE: {baseline_result.metrics['root_mean_squared_error']:.4f}")
print(f"winner   RMSE: {winner_result.metrics['root_mean_squared_error']:.4f}")

# The winner must improve RMSE over the baseline by at least 0.10 (abs).
mlflow.validate_evaluation_results(
    candidate_result=winner_result,
    baseline_result=baseline_result,
    validation_thresholds={
        "root_mean_squared_error": MetricThreshold(
            greater_is_better=False, min_absolute_change=0.10
        )
    },
)
print("\nGate cleared: winner beats the baseline by the required margin.")

2026/06/02 14:58:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/06/02 14:58:41 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/06/02 14:58:41 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-2482231eabd14c68814eae2ad6fa33e9


2026/06/02 14:58:41 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


2026/06/02 14:58:41 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


...

2026/06/02 14:58:43 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


2026/06/02 14:58:43 WARNING mlflow.models.evaluation.evaluators.shap: SHAP or matplotlib package is not installed, so model explainability insights will not be logged.


2026/06/02 14:58:43 INFO mlflow.models.evaluation.validation: Validating candidate model metrics against baseline


2026/06/02 14:58:43 INFO mlflow.models.evaluation.validation: Model validation passed!


🏃 View run winner_eval at: http://127.0.0.1:5001/#/experiments/8/runs/9998003ecd5e499ab14bd407cb29e04d
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/8
baseline RMSE: 1.1500
winner   RMSE: 0.5199

Gate cleared: winner beats the baseline by the required margin.


## Step 5 — Register the winner, promote it, keep the runner-up as challenger  ·  *reuses `g_`*

The winner that *passed the gate* becomes `@champion`; the runner-up becomes `@challenger` (the standing alternative for the next comparison). Production code will load `@champion` and never touch a version number — the decoupling `g_` teaches.

In [6]:
champion_version = mlflow.register_model(winner["model_uri"], MODEL_NAME).version
client.set_registered_model_alias(MODEL_NAME, "champion", champion_version)

challenger_version = mlflow.register_model(runner_up["model_uri"], MODEL_NAME).version
client.set_registered_model_alias(MODEL_NAME, "challenger", challenger_version)

# A little governance metadata, as in g_.
client.set_registered_model_tag(MODEL_NAME, "task", "regression")
client.set_model_version_tag(MODEL_NAME, champion_version, "validation_status", "approved")
client.update_model_version(
    MODEL_NAME, champion_version,
    description=f"Capstone champion ({winner['name']}): passed baseline gate.",
)

print(f"@champion   -> v{champion_version}  ({winner['name']})")
print(f"@challenger -> v{challenger_version}  ({runner_up['name']})")

Successfully registered model 'ca_housing_capstone'.
2026/06/02 14:58:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ca_housing_capstone, version 1


Created version '1' of model 'ca_housing_capstone'.


Registered model 'ca_housing_capstone' already exists. Creating a new version of this model...
2026/06/02 14:58:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ca_housing_capstone, version 2


Created version '2' of model 'ca_housing_capstone'.


@champion   -> v1  (rf_deep)
@challenger -> v2  (rf_mid)


## Step 6 — Load `@champion` and sanity-check  ·  *reuses `g_`*

Resolve the alias from scratch and predict on a few held-out rows — the same load path production code uses, confirming the promoted artifact is loadable and sane before we put a server in front of it.

In [7]:
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
sample = X_test.head()
local_preds = champion.predict(sample)
print(f"@champion resolves to v{client.get_model_version_by_alias(MODEL_NAME, 'champion').version}")
for idx, pred in zip(sample.index, local_preds):
    print(f"  row {idx:>5}: predicted {TARGET} = {pred:.3f}")

@champion resolves to v1
  row 14740: predicted MedHouseVal = 1.367
  row 10101: predicted MedHouseVal = 2.521
  row 20566: predicted MedHouseVal = 1.444
  row  2670: predicted MedHouseVal = 0.972
  row 15709: predicted MedHouseVal = 4.444


## Step 7 — Serve it for real, and call it over HTTP  ·  *uses `h_` (not re-taught)*

`h_` explains the *why* and the full `/invocations` contract; here we just **use** it as the capstone's final move. We stand up `mlflow models serve` on port 5002 as a background process (pointed at the alias, with `MLFLOW_TRACKING_URI` set so it resolves the registry), wait for `/health`, POST the **engineered** feature rows, and confirm the REST predictions match the in-process ones from Step 6 — then shut the server down.

In real life you'd run the serve command in its own terminal (as `h_` shows); driving it from a subprocess here just keeps the capstone self-contained and runnable end to end.

In [8]:
serve_cmd = [
    "mlflow", "models", "serve",
    "-m", f"models:/{MODEL_NAME}@champion",
    "--host", "127.0.0.1", "--port", "5002",
    "--env-manager", "local",
]
env = {**os.environ, "MLFLOW_TRACKING_URI": TRACKING_URI}
proc = subprocess.Popen(serve_cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
try:
    # Wait for the server to load the model and report healthy.
    up = False
    for _ in range(90):
        try:
            if requests.get(f"{SERVE_URL}/health", timeout=2).status_code == 200:
                up = True
                break
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    print("serving healthy:", up)

    payload = {"dataframe_split": {"columns": list(sample.columns),
                                   "data": sample.values.tolist()}}
    resp = requests.post(f"{SERVE_URL}/invocations", json=payload, timeout=30)
    resp.raise_for_status()
    rest_preds = resp.json()["predictions"]

    comparison = pd.DataFrame({
        "in_process": np.round(local_preds, 6),
        "over_REST": np.round(rest_preds, 6),
    })
    print(comparison.to_string(index=False))
    print("\nidentical:", bool((comparison["in_process"] == comparison["over_REST"]).all()))
finally:
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("serving process stopped")

serving healthy: True
 in_process  over_REST
   1.367243   1.367243
   2.520756   2.520756
   1.443649   1.443649
   0.972358   0.972358
   4.444487   4.444487

identical: True
serving process stopped


That closing equality is the whole pipeline paying off: the *same* artifact we engineered features for, tuned, evaluated, gated, registered, and promoted is now answering HTTP requests with identical numbers to the in-process model. One dataset, one model, end to end.

## The "Traces" tab — orientation (so the empty menu doesn't puzzle you)

In the MLflow 3 UI you'll have noticed a **Traces** tab next to Runs and Models — and for everything in this repo it's **empty**. That's expected, not a misconfiguration:

- **What it is.** MLflow *Tracing* records OpenTelemetry-style **spans** (inputs/outputs/latency for each step) — built for **GenAI / LLM apps and agents** (LangChain, OpenAI, LlamaIndex, …), where you need to see *inside* a chain or agent call.
- **Why it's empty here.** Traditional-ML runs (sklearn/XGBoost `fit`/`predict`) emit no spans. Nothing in this repo produces traces.
- **Where it lights up.** Instrumenting GenAI code with `@mlflow.trace` or a GenAI autolog flavor — the **GenAI track**, which is intentionally out of scope for this traditional-ML repo.

This note exists purely for **MLflow-3 UI literacy**: now you know what the tab is and why your work doesn't fill it.

## Where this leaves you

You've now run the full traditional-ML MLOps spine on one dataset:

| Stage | This notebook | Deep dive |
|---|---|---|
| Data lineage | Step 1 | `i_dataset_logging` |
| Tuning + resource cost | Step 2 | `c_hyperparameter_tuning`, `j_system_metrics` |
| Evaluation | Step 3 | `f_model_evaluation` |
| Validation gate | Step 4 | `f_model_evaluation` |
| Registry + promotion | Step 5–6 | `g_model_registry` |
| Serving | Step 7 | `h_model_serving` |

**Beyond this repo:** packaging with MLflow Projects (`MLproject`), a team setup (remote backend + artifact store), deployment targets (SageMaker/K8s), and the **GenAI track** (tracing, `mlflow.genai.evaluate`) that the Traces tab hints at.